## Imports and settings

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             RocCurveDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
FIG_DIR = Path("../reports/figures")
MODEL_DIR = Path("../models")
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Clean the data

In [3]:
df = pd.read_csv(DATA_PATH)

# The 11 blank TotalCharges belong to tenure-0 customers who haven't been billed yet -> 0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

# customerID is an identifier with no predictive value
df = df.drop(columns=["customerID"])

# Target: Yes/No -> 1/0
df["Churn"] = (df["Churn"] == "Yes").astype(int)

X = df.drop(columns=["Churn"])
y = df["Churn"]

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]
print(X.shape, "| numeric:", len(num_cols), "| categorical:", len(cat_cols))

(7043, 19) | numeric: 3 | categorical: 16


In [4]:
# Stratified so both sets keep the 26.5% churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "churn rate", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "churn rate", round(y_test.mean(), 3))

Train: (5634, 19) churn rate 0.265
Test:  (1409, 19) churn rate 0.265


In [5]:
# Scale numeric features (needed for KNN distances and LR coefficients)
# One-hot encode categoricals; binary Yes/No columns become a single 0/1 column
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
])
print("Features after encoding:", preprocessor.fit(X_train).transform(X_train).shape[1])

Features after encoding: 40


In [6]:
def evaluate(name, setting, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    return {"Model": name, "Setting": setting,
            "Accuracy": accuracy_score(y_te, y_pred),
            "Precision": precision_score(y_te, y_pred),
            "Recall": recall_score(y_te, y_pred),
            "F1": f1_score(y_te, y_pred),
            "ROC-AUC": roc_auc_score(y_te, y_prob)}

models = {
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
}

In [7]:
results = []
baseline_models = {}

for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    baseline_models[name] = pipe
    results.append(evaluate(name, "Baseline", pipe, X_test, y_test))

pd.DataFrame(results).round(3)

,Model,Setting,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN,Baseline,0.762,0.554,0.532,0.543,0.777
1,Logistic Regression,Baseline,0.806,0.657,0.559,0.604,0.842
2,Random Forest,Baseline,0.781,0.612,0.481,0.539,0.821


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "KNN": {"clf__n_neighbors": [5, 11, 21, 31, 51],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2]},                       # 1 = Manhattan, 2 = Euclidean
    "Logistic Regression": {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    "Random Forest": {"clf__n_estimators": [200, 400],
                      "clf__max_depth": [5, 10, None],
                      "clf__min_samples_leaf": [1, 5, 10],
                      "clf__max_features": ["sqrt", 0.5]},
}

tuned_models = {}
for name, clf in models.items():
    # SMOTE sits INSIDE the pipeline, so it only oversamples the training folds,
    # never the validation fold or the test set (prevents data leakage)
    pipe = ImbPipeline([("prep", preprocessor),
                        ("smote", SMOTE(random_state=RANDOM_STATE)),
                        ("clf", clf)])
    gs = GridSearchCV(pipe, param_grids[name], scoring="f1", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned_models[name] = gs.best_estimator_
    print(f"{name}: best params {gs.best_params_}, CV F1 = {gs.best_score_:.3f}")
    results.append(evaluate(name, "Tuned + SMOTE", gs.best_estimator_, X_test, y_test))

KNN: best params {'clf__n_neighbors': 31, 'clf__p': 1, 'clf__weights': 'uniform'}, CV F1 = 0.614
Logistic Regression: best params {'clf__C': 0.1}, CV F1 = 0.633
Random Forest: best params {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 10, 'clf__n_estimators': 200}, CV F1 = 0.636


In [9]:
results_df = pd.DataFrame(results).set_index(["Model", "Setting"]).round(3)
results_df.to_csv("../reports/model_results.csv")
results_df

,,Accuracy,Precision,Recall,F1,ROC-AUC
Model,Setting,,,,,
KNN,Baseline,0.762,0.554,0.532,0.543,0.777
Logistic Regression,Baseline,0.806,0.657,0.559,0.604,0.842
Random Forest,Baseline,0.781,0.612,0.481,0.539,0.821
KNN,Tuned + SMOTE,0.710,0.472,0.799,0.594,0.822
Logistic Regression,Tuned + SMOTE,0.737,0.503,0.786,0.614,0.839
Random Forest,Tuned + SMOTE,0.761,0.537,0.722,0.616,0.838
